In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import psycopg2
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_collection.logging_config import logger

/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
verolizer = {
    'positive': ['buy'],
    'negative': ['sell'],
}

model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "12GiB", "cpu": "64GiB"},  
)

model_driver = ModelDriver(model_name=model_name, model=model, verbolizer=verolizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.12it/s]
2026-05-24 14:35:50,154 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [3]:
report = model_driver.get_random_reports(seed=45)[0]

In [4]:
print(report[:500])

UNITED STATESSECURITIES AND EXCHANGE COMMISSIONWashington, D.C. 20549FORM 10-Q (MARK ONE)QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934For the quarterly period ended May 1, 2022ORTRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934For the transition period from                      to                     Broadcom Inc.(Exact name of registrant as specified in its charter)Delaware001-3844935-2617337(State or other jurisdi


In [5]:
len(report)

219378

In [6]:
#report = report[:500]

good promts:

"From this report, the filing cadence is: annual or quarterly? It is"

In [7]:
# Report type prompts
report_type_prompts = [
    "Question: This SEC filing is annual or quarterly? Answer with one word: annual or quarterly. My answer is:",    
    "Classify filing type. Output one word only: annual | quarterly. Output:",
    "From this report, the filing cadence is: annual or quarterly? It is" # Best one
    ]

# Filed quarter prompts 
fq_prompts = [
    "Use the reporting period end date in this SEC filing to determine the calendar quarter. Select one category: first, second, third, or fourth. Selected category:",
    "Classify the calendar quarter implied by the phrase about the period ended date. Choose the best label from first, second, third, fourth. Best label:",
    "Ignore filing date and fiscal quarter numbering; use only the period end date. Among first, second, third, and fourth, provide the most appropriate category. Category:",
]

# Sector prompts 
sector_prompts = [
    "Classify company sector from this filing. Output one word from: technology, industrials, financials, healthcare, energy, utilities, materials, communication, cyclical, defensive, realestate.",
    "What is the company sector? One-word label only from the allowed list.",
    "Sector prediction task. Return exactly one allowed sector token."
    ]

# Market cup bucket prompts
mc_prompts = [
    "Estimate market-cap class from this filing. Output one word: micro, small, mid, large, or mega.",
    "Classify market cap bucket. One-word output only: micro|small|mid|large|mega.",
    "Company size by market cap is: micro/small/mid/large/mega (one word)."
    ]

In [8]:
def check_prompts (prompts, verbolizer):
    for prompt in prompts:
        print(model_driver.predict_metadata(text=report, prompt=prompt, verbolizer=verbolizer, print_top_tokens=True, top_k_tokens=15))

In [9]:
report_type_verbolizer = {
    "10-K": ["annual"],
    "10-Q": ["quarter"],
}

fq_verbolizer = {
    "first": [
        "first", "january", "february", "march",
        "spring", "earlyyear"
    ],
    "second": [
        "second", "april", "may", "june",
        "midyear"
    ],
    "third": [
        "third", "july", "august", "september",
        "lateyear"
    ],
    "fourth": [
        "fourth", "october", "november", "december",
        "yearend", "annual", "yearly"
    ],
}

sector_verbolizer = {
    "Technology": ["technology", "tech"],
    "Healthcare": ["healthcare", "medical"],
    "Financial Services": ["financial", "banking"],
    "Consumer Cyclical": ["cyclical", "consumer"],
    "Consumer Defensive": ["defensive", "staples"],
    "Industrials": ["industrials", "industrial"],
    "Energy": ["energy", "oil"],
    "Utilities": ["utilities", "utility"],
    "Real Estate": ["realestate", "property"],
    "Basic Materials": ["materials", "mining"],
    "Communication Services": ["communication", "media"],
}

mc_verbolizer = {
    "micro": ["micro", "tiny"],
    "small": ["small"],
    "mid": ["mid", "medium"],
    "large": ["large", "big"],
    "mega": ["mega", "giant"],
}


In [10]:
check_prompts(prompts=report_type_prompts, verbolizer=report_type_verbolizer)


Top 15 tokens by probability:
annual       | 0.490234
quarter      | 0.261719
Annual       | 0.027710
Quarter      | 0.016724
it           | 0.005127
fiscal       | 0.003998
Both         | 0.003632
AN           | 0.003006
Qu           | 0.002731
semi         | 0.002350
annually     | 0.002136
neither      | 0.001823
Answer       | 0.001610
qu           | 0.001465
https        | 0.001076
{'confidence': 0.79638671875, 'probabilities': {'10-K': 0.6503678724708768, '10-Q': 0.34963212752912326}}

Top 15 tokens by probability:
annual       | 0.047119
cond         | 0.031494
Annual       | 0.031494
quarter      | 0.020264
fiscal       | 0.015320
all          | 0.011536
financial    | 0.009277
one          | 0.008179
Cond         | 0.008179
Table        | 0.007935
HTML         | 0.006592
Financial    | 0.006195
text         | 0.005798
Quarter      | 0.005646
as           | 0.004517
{'confidence': 0.104522705078125, 'probabilities': {'10-K': 0.7521167883211679, '10-Q': 0.24788321167883212}}

T

In [11]:
#check_prompts(prompts=fq_prompts, verbolizer=fq_verbolizer)

In [12]:
#check_prompts(prompts=sector_prompts, verbolizer=sector_verbolizer)

In [13]:
#check_prompts(prompts=mc_prompts, verbolizer=sector_verbolizer)